# 05a · Qdrant data model and CRUD

## The engineering question
What exactly is stored in a Qdrant collection, and which changes are schema changes?

**Learning outcomes:** distinguish a collection, a point ID, named vector fields, and payload; execute create/read/update/delete; observe dimensionality validation; explain idempotent upsert.

A point holds an ID, vector representations, and metadata payload. A named vector field fixes its representation's dimensionality and metric. Payload stores attributes such as tenant or source revision; it is not an authorization system by itself. Reusing a point ID replaces that point on upsert, while generating a new ID creates a different point.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("05a", "T", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
from uuid import uuid4
from qdrant_client import QdrantClient, models as qm
from qdrant_client.http.exceptions import UnexpectedResponse
from vector_masterclass.config import qdrant_url

client = QdrantClient(url=qdrant_url(), timeout=10)
name = "vmc_05a_" + uuid4().hex[:8]
events = []
client.create_collection(
    name,
    vectors_config={"dense": qm.VectorParams(size=3, distance=qm.Distance.COSINE)},
    sparse_vectors_config={"lexical": qm.SparseVectorParams()},
)
try:
    points = [
        qm.PointStruct(
            id=1, vector={"dense": [1.0, 0.0, 0.0]}, payload={"tenant": "red", "revision": 1}
        ),
        qm.PointStruct(
            id=2, vector={"dense": [0.0, 1.0, 0.0]}, payload={"tenant": "blue", "revision": 1}
        ),
    ]
    client.upsert(name, points=points, wait=True)
    events.append({"operation": "insert two points", "count": client.count(name, exact=True).count})
    client.upsert(name, points=points, wait=True)
    events.append(
        {"operation": "repeat identical upsert", "count": client.count(name, exact=True).count}
    )
    client.set_payload(name, payload={"revision": 2}, points=[1], wait=True)
    revised = client.retrieve(name, ids=[1])[0]
    dimension_rejected = False
    try:
        client.upsert(name, points=[qm.PointStruct(id=3, vector={"dense": [1.0, 0.0]})], wait=True)
    except UnexpectedResponse as error:
        dimension_rejected = error.status_code == 400
        print("Expected dimensionality rejection:", error.status_code)
    client.delete(name, points_selector=qm.PointIdsList(points=[2]), wait=True)
    events.append({"operation": "delete point 2", "count": client.count(name, exact=True).count})
    absent = client.retrieve(name, ids=[2])
finally:
    client.delete_collection(name)
    client.close()
results = lesson.table(pd.DataFrame(events))
display(results)
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(results.operation, results["count"], color="#2563eb")
ax.set(xlabel="Stored point count", title="Stable IDs make retries idempotent")
ax.set_xticks([0, 1, 2])
fig.tight_layout()
lesson.figure(fig)
plt.show()

## Failure and exercise
A two-dimensional vector cannot enter a three-dimensional field. A subtler error remains possible: a different model can produce the same dimensions. The application must bind model/configuration identity to the index; server dimension checks cannot establish semantic compatibility.

**Exercise:** repeat the upsert using fresh IDs and observe the duplicate logical records. Propose a stable ID derived from document revision and source span. Explain why the same tenant string in a payload does not stop a client from querying another tenant.

Define vector fields when creating a collection. Adding an unconfigured sparse field through an ordinary collection update failed in our pinned-server pilot; later migrations use explicit schema contracts. Use named representations for deliberate hybrid pipelines, not for mixing unrelated spaces. Read/write acknowledgment, persistence, and background indexing are separated in lesson 05b.

In [ ]:
manifest = lesson.finish(
    {
        "idempotent_upsert": results["count"].tolist() == [2, 2, 1],
        "payload_update_visible": revised.payload["revision"] == 2,
        "dimension_mismatch_rejected": dimension_rejected,
        "delete_visible": len(absent) == 0,
    }
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.